In [1]:
import tensorflow as tf
from tqdm import tqdm_notebook

import recora.data.data_info

print(tf.config.list_physical_devices('GPU'))

C:\Users\PC\.conda\envs\recora\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [18]:
# ============================================================
# 0) INSTALL
# ============================================================
# pip install -U LibRecommender tensorflow pandas numpy clickhouse-connect scikit-learn

# If you are in notebook / interactive env and hit TF graph reuse issues:
# import tensorflow as tf
# tf.compat.v1.reset_default_graph()


# ============================================================
# 1) IMPORTS
# ============================================================
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import clickhouse_connect
from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer

from recora.data import DatasetFeat

import pandas as pd

from recora.data import DatasetFeat, split_by_ratio_chrono, random_split

TRAIN_DAYS = 120
RANDOM_SEED = 42
TOP_K = 20

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

warnings.filterwarnings("ignore")

In [107]:
# ============================================================
# 2) GLOBAL CONFIG
# ============================================================
CLICKHOUSE_CONFIG = {
	"host": os.getenv("CH_HOST", "172.30.36.13"),
	"port": int(os.getenv("CH_PORT", "8123")),
	"username": os.getenv("CH_USER", "default"),
	"password": os.getenv("CH_PASSWORD", "192837465AhurA!@#"),
	"database": os.getenv("CH_DATABASE", "KF"),
	"secure": os.getenv("CH_SECURE", "false").lower() == "true",
}

# ============================================================
# 3) CLICKHOUSE CLIENT
# ============================================================
client = clickhouse_connect.get_client(**CLICKHOUSE_CONFIG)
#
# INTERACTION_SQL = f"""
# WITH
#     /* ============================================================
#     1) BASE POSITIVE EVENTS
#     Grain:
#     one row per (user_id, event_date, event_hour, target_category_id)
#     ============================================================ */
#     base_events AS (SELECT oi.user_id                                                    AS user_id,
#                            oi.order_date                                                 AS event_date,
#                            oi.order_hour                                                 AS event_hour,
#                            toDateTime(oi.order_date) + toIntervalHour(oi.order_hour)     AS event_ts,
#
#                            pp.category_level2_id                                         AS target_category_id,
#                            pp.category_level1_id                                         AS target_root_category_id,
#
#                            sum(oi.quantity)                                              AS target_qty,
#
#                            sum(
#                                    oi.payable_price
#                                        + oi.items_voucher_discount
#                                        + oi.items_jet_discount
#                            )                                                             AS target_spend,
#
#                            sum(
#                                    oi.items_jet_discount
#                                        + oi.brand_discount
#                                        + oi.items_vendor_discount
#                            )                                                             AS target_discount_value,
#
#                            exp(-log(2) / 14.0 * dateDiff('day', oi.order_date, today())) AS sample_weight,
#                            1                                                             AS label
#                     FROM KF.fact_order_items oi
#                              INNER JOIN KF.plane_products pp
#                                         ON oi.shop_product_id = pp.shop_product_id
#                     WHERE oi.is_gross = 1
#                       AND oi.order_date > today() - {TRAIN_DAYS}
#                       AND pp.category_level1_activity_type_id IN (1, 29)
#                       AND pp.category_level2_activity_type_id IN (1, 29)
#                       AND pp.product_id <> 0
#                     GROUP BY oi.user_id,
#                              oi.order_date,
#                              oi.order_hour,
#                              pp.category_level2_id,
#                              pp.category_level1_id),
#
#     /* ============================================================
#     2) CONTEXT FEATURES
#     ============================================================ */
#     context_features AS (SELECT b.user_id,
#                                 b.event_date,
#                                 b.event_hour,
#                                 b.event_ts,
#                                 b.target_category_id,
#                                 b.target_root_category_id,
#
#                                 toDayOfWeek(b.event_date)                     AS dow,
#                                 if(toDayOfWeek(b.event_date) IN (4, 5), 1, 0) AS is_weekend,
#
#                                 multiIf(
#                                         b.event_hour < 12, 1,
#                                         b.event_hour < 18, 2,
#                                         3
#                                 )                                             AS hour_bucket
#                          FROM base_events b),
#
#     /* ============================================================
#     3) USER HISTORY FEATURES
#     Event-level: one row per current event
#     ============================================================ */
#     user_hist AS (SELECT cur.user_id,
#                          cur.event_date,
#                          cur.event_hour,
#                          cur.event_ts,
#                          cur.target_category_id,
#                          cur.target_root_category_id,
#
#                          countIf(hist.event_ts < cur.event_ts) AS u_hist_orders,
#
#                          countDistinctIf(
#                                  hist.target_category_id,
#                                  hist.event_ts < cur.event_ts
#                          )                                     AS u_hist_categories,
#
#                          countDistinctIf(
#                                  hist.target_root_category_id,
#                                  hist.event_ts < cur.event_ts
#                          )                                     AS u_hist_roots,
#
#                          sumIf(
#                                  hist.target_qty,
#                                  hist.event_ts < cur.event_ts
#                          )                                     AS u_hist_qty,
#
#                          sumIf(
#                                  hist.target_spend,
#                                  hist.event_ts < cur.event_ts
#                          )                                     AS u_hist_spend,
#
#                          sumIf(
#                                  hist.target_discount_value,
#                                  hist.event_ts < cur.event_ts
#                          )                                     AS u_hist_discount_value,
#
#                          dateDiff(
#                                  'day',
#                                  minIf(hist.event_date, hist.event_ts < cur.event_ts),
#                                  cur.event_date
#                          )                                     AS u_days_since_first_order,
#
#                          dateDiff(
#                                  'day',
#                                  maxIf(hist.event_date, hist.event_ts < cur.event_ts),
#                                  cur.event_date
#                          )                                     AS u_days_since_last_order,
#
#                          countIf(
#                                  hist.event_ts < cur.event_ts
#                                      AND hist.event_date >= cur.event_date - 7
#                          )                                     AS u_orders_7d,
#
#                          countIf(
#                                  hist.event_ts < cur.event_ts
#                                      AND hist.event_date >= cur.event_date - 30
#                          )                                     AS u_orders_30d,
#
#                          sumIf(
#                                  hist.target_spend,
#                                  hist.event_ts < cur.event_ts
#                                      AND hist.event_date >= cur.event_date - 7
#                          )                                     AS u_spend_7d,
#
#                          sumIf(
#                                  hist.target_spend,
#                                  hist.event_ts < cur.event_ts
#                                      AND hist.event_date >= cur.event_date - 30
#                          )                                     AS u_spend_30d,
#
#                          uniqExactIf(
#                                  hist.event_date,
#                                  hist.event_ts < cur.event_ts
#                                      AND hist.event_date >= cur.event_date - 30
#                          )                                     AS u_active_days_30d,
#
#                          avgIf(
#                                  hist.event_hour,
#                                  hist.event_ts < cur.event_ts
#                          )                                     AS u_avg_order_hour,
#
#                          avgIf(
#                                  hist.target_discount_value / nullIf(hist.target_spend + hist.target_discount_value, 0),
#                                  hist.event_ts < cur.event_ts
#                          )                                     AS u_avg_discount_ratio_hist
#                   FROM base_events cur
#                            LEFT JOIN base_events hist
#                                      ON cur.user_id = hist.user_id
#                   GROUP BY cur.user_id,
#                            cur.event_date,
#                            cur.event_hour,
#                            cur.event_ts,
#                            cur.target_category_id,
#                            cur.target_root_category_id),
#
#     /* ============================================================
#     4) USER-CATEGORY HISTORY FEATURES
#     ============================================================ */
#     user_category_hist AS (SELECT cur.user_id,
#                                   cur.event_date,
#                                   cur.event_hour,
#                                   cur.event_ts,
#                                   cur.target_category_id,
#                                   cur.target_root_category_id,
#
#                                   countIf(
#                                           hist.event_ts < cur.event_ts
#                                               AND hist.target_category_id = cur.target_category_id
#                                   ) AS uc_hist_orders,
#
#                                   sumIf(
#                                           hist.target_qty,
#                                           hist.event_ts < cur.event_ts
#                                               AND hist.target_category_id = cur.target_category_id
#                                   ) AS uc_hist_qty,
#
#                                   sumIf(
#                                           hist.target_spend,
#                                           hist.event_ts < cur.event_ts
#                                               AND hist.target_category_id = cur.target_category_id
#                                   ) AS uc_hist_spend,
#
#                                   countIf(
#                                           hist.event_ts < cur.event_ts
#                                               AND hist.target_category_id = cur.target_category_id
#                                               AND hist.event_date >= cur.event_date - 7
#                                   ) AS uc_orders_7d,
#
#                                   countIf(
#                                           hist.event_ts < cur.event_ts
#                                               AND hist.target_category_id = cur.target_category_id
#                                               AND hist.event_date >= cur.event_date - 30
#                                   ) AS uc_orders_30d,
#
#                                   countIf(
#                                           hist.event_ts < cur.event_ts
#                                               AND hist.target_category_id = cur.target_category_id
#                                               AND hist.event_date >= cur.event_date - 90
#                                   ) AS uc_orders_90d,
#
#                                   sumIf(
#                                           hist.target_spend,
#                                           hist.event_ts < cur.event_ts
#                                               AND hist.target_category_id = cur.target_category_id
#                                               AND hist.event_date >= cur.event_date - 30
#                                   ) AS uc_spend_30d,
#
#                                   dateDiff(
#                                           'day',
#                                           minIf(
#                                                   hist.event_date,
#                                                   hist.event_ts < cur.event_ts
#                                                       AND hist.target_category_id = cur.target_category_id
#                                           ),
#                                           cur.event_date
#                                   ) AS uc_days_since_first,
#
#                                   dateDiff(
#                                           'day',
#                                           maxIf(
#                                                   hist.event_date,
#                                                   hist.event_ts < cur.event_ts
#                                                       AND hist.target_category_id = cur.target_category_id
#                                           ),
#                                           cur.event_date
#                                   ) AS uc_days_since_last,
#
#                                   if(
#                                           countIf(
#                                                   hist.event_ts < cur.event_ts
#                                                       AND hist.target_category_id = cur.target_category_id
#                                           ) > 0,
#                                           1,
#                                           0
#                                   ) AS uc_is_repeat,
#
#                                   sumIf(
#                                           exp(-log(2) / 14.0 * dateDiff('day', hist.event_date, cur.event_date)),
#                                           hist.event_ts < cur.event_ts
#                                               AND hist.target_category_id = cur.target_category_id
#                                   ) AS uc_decayed_count
#                            FROM base_events cur
#                                     LEFT JOIN base_events hist
#                                               ON cur.user_id = hist.user_id
#                            GROUP BY cur.user_id,
#                                     cur.event_date,
#                                     cur.event_hour,
#                                     cur.event_ts,
#                                     cur.target_category_id,
#                                     cur.target_root_category_id),
#
#     /* ============================================================
#     5) FINAL
#     ============================================================ */
#     final AS (SELECT b.user_id                                       AS user,
#                      b.event_date                                    AS event_date,
#                      b.event_hour                                    AS event_hour,
#                      b.event_ts                                      AS time,
#
#                      b.sample_weight                                 AS sample_weight,
#                      b.label                                         AS label,
#
#                      b.target_category_id                            AS item,
#                      b.target_root_category_id                       AS root_category_id,
#
#                      b.target_qty                                    AS target_qty,
#                      b.target_spend                                  AS target_spend,
#                      b.target_discount_value                         AS target_discount_value,
#
#
#                      c.dow                                           AS dow,
#                      c.is_weekend                                    AS is_weekend,
#                      c.hour_bucket                                   AS hour_bucket,
#
#                      uh.u_hist_orders                                AS u_hist_orders,
#                      uh.u_hist_categories                            AS u_hist_categories,
#                      uh.u_hist_roots                                 AS u_hist_roots,
#                      uh.u_hist_qty                                   AS u_hist_qty,
#                      uh.u_hist_spend                                 AS u_hist_spend,
#                      uh.u_hist_discount_value                        AS u_hist_discount_value,
#                      uh.u_days_since_first_order                     AS u_days_since_first_order,
#                      uh.u_days_since_last_order                      AS u_days_since_last_order,
#                      uh.u_orders_7d                                  AS u_orders_7d,
#                      uh.u_orders_30d                                 AS u_orders_30d,
#                      uh.u_spend_7d                                   AS u_spend_7d,
#                      uh.u_spend_30d                                  AS u_spend_30d,
#                      uh.u_active_days_30d                            AS u_active_days_30d,
#                      uh.u_avg_order_hour                             AS u_avg_order_hour,
#                      uh.u_avg_discount_ratio_hist                    AS u_avg_discount_ratio_hist,
#
#                      uh.u_hist_spend / nullIf(uh.u_hist_orders, 0)   AS u_avg_spend_per_order_hist,
#                      uh.u_spend_30d / nullIf(uh.u_orders_30d, 0)     AS u_avg_spend_per_order_30d,
#
#                      uc.uc_hist_orders                               AS uc_hist_orders,
#                      uc.uc_hist_qty                                  AS uc_hist_qty,
#                      uc.uc_hist_spend                                AS uc_hist_spend,
#                      uc.uc_orders_7d                                 AS uc_orders_7d,
#                      uc.uc_orders_30d                                AS uc_orders_30d,
#                      uc.uc_orders_90d                                AS uc_orders_90d,
#                      uc.uc_spend_30d                                 AS uc_spend_30d,
#                      uc.uc_days_since_first                          AS uc_days_since_first,
#                      uc.uc_days_since_last                           AS uc_days_since_last,
#                      uc.uc_is_repeat                                 AS uc_is_repeat,
#                      uc.uc_decayed_count                             AS uc_decayed_count,
#
#                      uc.uc_hist_spend / nullIf(uc.uc_hist_orders, 0) AS uc_avg_spend_hist,
#                      uc.uc_spend_30d / nullIf(uc.uc_orders_30d, 0)   AS uc_avg_spend_30d,
#                      uc.uc_orders_30d / nullIf(uh.u_orders_30d, 0)   AS uc_order_share_30d,
#                      uc.uc_spend_30d / nullIf(uh.u_spend_30d, 0)     AS uc_spend_share_30d,
#
#                      abs(b.event_hour - uh.u_avg_order_hour)         AS diff_user_avg_order_hour
#
#               FROM base_events b
#                        LEFT JOIN context_features c
#                                  ON b.user_id = c.user_id
#                                      AND b.event_date = c.event_date
#                                      AND b.event_hour = c.event_hour
#                                      AND b.event_ts = c.event_ts
#                                      AND b.target_category_id = c.target_category_id
#                                      AND b.target_root_category_id = c.target_root_category_id
#
#                        LEFT JOIN user_hist uh
#                                  ON b.user_id = uh.user_id
#                                      AND b.event_date = uh.event_date
#                                      AND b.event_hour = uh.event_hour
#                                      AND b.event_ts = uh.event_ts
#                                      AND b.target_category_id = uh.target_category_id
#                                      AND b.target_root_category_id = uh.target_root_category_id
#
#                        LEFT JOIN user_category_hist uc
#                                  ON b.user_id = uc.user_id
#                                      AND b.event_date = uc.event_date
#                                      AND b.event_hour = uc.event_hour
#                                      AND b.event_ts = uc.event_ts
#                                      AND b.target_category_id = uc.target_category_id
#                                      AND b.target_root_category_id = uc.target_root_category_id)
#
# SELECT *
# FROM final
# """

INTERACTION_SQL = f"""
WITH recent_users AS (
    SELECT user_id
    FROM KF.fact_orders
    WHERE order_date BETWEEN today() - {TRAIN_DAYS} AND yesterday()
    GROUP BY user_id
)

SELECT
    -- item
    concat(pp.category_level2_id, '098', brand_id)  AS item,
    pp.category_level2_id                           AS category_level2_id,
    pp.category_level1_id                           AS category_level1_id,
    pp.brand_id                                     AS brand_id,

    -- user
    oi.user_id                                      AS user,
    ms.merchant_shop_polygon_id                     AS polygon_id,

    -- ── USER × ITEM FEATURES ──────────────────────────────────────────────
    count(*) OVER (
        PARTITION BY oi.user_id, concat(pp.category_level2_id, '098', pp.brand_id)
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS user_item_frequency,

    count(*) OVER (
        PARTITION BY oi.user_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS user_frequency,

    dateDiff('day',
        lagInFrame(oi.order_date, 1) OVER (
            PARTITION BY oi.user_id, concat(pp.category_level2_id, '098', pp.brand_id)
            ORDER BY oi.order_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), oi.order_date
    )                                               AS user_item_recency,

    dateDiff('day',
        lagInFrame(oi.order_date, 1) OVER (
            PARTITION BY oi.user_id
            ORDER BY oi.order_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), oi.order_date
    )                                               AS user_recency,

    -- ── USER × CATEGORY FEATURES ─────────────────────────────────────────
    count(*) OVER (
        PARTITION BY oi.user_id, pp.category_level2_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS user_cat2_frequency,

    count(*) OVER (
        PARTITION BY oi.user_id, pp.category_level1_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS user_cat1_frequency,

    dateDiff('day',
        lagInFrame(oi.order_date, 1) OVER (
            PARTITION BY oi.user_id, pp.category_level2_id
            ORDER BY oi.order_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), oi.order_date
    )                                               AS user_cat2_recency,

    -- ── USER × BRAND FEATURES ────────────────────────────────────────────
    count(*) OVER (
        PARTITION BY oi.user_id, pp.brand_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS user_brand_frequency,

    dateDiff('day',
        lagInFrame(oi.order_date, 1) OVER (
            PARTITION BY oi.user_id, pp.brand_id
            ORDER BY oi.order_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), oi.order_date
    )                                               AS user_brand_recency,

    -- ── USER × POLYGON FEATURES ──────────────────────────────────────────
    count(*) OVER (
        PARTITION BY oi.user_id, ms.merchant_shop_polygon_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS user_polygon_frequency,

    -- ── ITEM GLOBAL POPULARITY (polygon-aware) ───────────────────────────
    count(*) OVER (
        PARTITION BY concat(pp.category_level2_id, '098', pp.brand_id)
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS item_global_frequency,

    count(*) OVER (
        PARTITION BY concat(pp.category_level2_id, '098', pp.brand_id),
                     ms.merchant_shop_polygon_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS item_polygon_frequency,

    count(*) OVER (
        PARTITION BY pp.category_level2_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS cat2_global_frequency,

    count(*) OVER (
        PARTITION BY pp.brand_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS brand_global_frequency,

    -- ── USER BASKET BEHAVIOR ─────────────────────────────────────────────
    -- distinct items bought by user so far (catalog coverage)
    count(DISTINCT concat(pp.category_level2_id, '098', pp.brand_id)) OVER (
        PARTITION BY oi.user_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS user_distinct_items,

    -- distinct categories explored
    count(DISTINCT pp.category_level2_id) OVER (
        PARTITION BY oi.user_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS user_distinct_cat2,

    -- distinct brands explored
    count(DISTINCT pp.brand_id) OVER (
        PARTITION BY oi.user_id
        ORDER BY oi.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                               AS user_distinct_brands,

    -- ── TIME FEATURES ────────────────────────────────────────────────────
    oi.order_date                                   AS time,
    toDayOfWeek(oi.order_date)                      AS day_of_week,       -- 1=Mon … 7=Sun
    toISOWeek(oi.order_date)                        AS week_of_year,
    if(toDayOfWeek(oi.order_date) >= 6, 1, 0)      AS is_weekend,

    multiIf(
        oi.order_hour BETWEEN 7  AND 11, '1',
        oi.order_hour BETWEEN 12 AND 16, '2',
        oi.order_hour BETWEEN 17 AND 21, '3',
        oi.order_hour BETWEEN 22 AND 24, '4',
        '0'
    )                                               AS hour_bucket,

    -- ── SAMPLE WEIGHT & LABEL ────────────────────────────────────────────
    exp(-log(2) / 14.0 * dateDiff('day', oi.order_date, today())) AS sample_weight,
    1                                                              AS label

FROM KF.fact_order_items oi
JOIN KF.plane_products pp
    ON  oi.shop_product_id = pp.shop_product_id
    AND oi.is_gross = 1
    AND oi.order_date BETWEEN today() - {TRAIN_DAYS} AND yesterday()
    AND pp.category_level1_activity_type_id IN (1, 29)
    AND pp.category_level2_activity_type_id IN (1, 29)
    AND pp.product_id          <> 0
    AND pp.brand_id            <> 0
    AND pp.category_level2_id  <> 0
    AND pp.category_level1_id  <> 0
    AND oi.items_jet_discount   = 0
JOIN KF.dim_merchant_shops ms
    ON  oi.merchant_shop_id = ms.merchant_shop_id
    AND ms.activity_type_id IN (1, 29)

"""

inter_events = client.query_df(INTERACTION_SQL)
print("inter_events shape :", inter_events.shape)

inter_events shape : (5840243, 30)


In [108]:
inter_events.to_parquet("time_series_recora_category_inter_events.parquet", engine="pyarrow", index=False)

In [109]:
inter_events = pd.read_parquet("time_series_recora_category_inter_events.parquet", engine="pyarrow")

In [110]:
inter_events['user'] = inter_events['user'].astype(str)
inter_events['item'] = inter_events['item'].astype(str)

In [111]:
inter_events = inter_events[inter_events['user'] != 20524543]

In [112]:
inter_events.loc[(inter_events['user_recency'] > 120) | (inter_events['user_item_recency'] > 120), ['user_recency', 'user_item_recency']] = 120

In [114]:
pareto_items = inter_events.groupby('item').size().rename("pareto").sort_values(ascending=False).cumsum()
pareto_items = pareto_items / pareto_items.max()
pareto_items = pareto_items[pareto_items <= 1.0].reset_index()

pareto_users = inter_events.groupby('user').size()
pareto_users = pareto_users / pareto_users.max()
pareto_users = pareto_users[pareto_users <= 1.0].reset_index()

In [115]:
inter_events = inter_events.loc[
	(inter_events['item'].isin(pareto_items['item'])) &
	(inter_events['user'].isin(pareto_users['user']))
	]

In [116]:
inter_events = inter_events.sort_values(by='time', ascending=True)

In [257]:
from recora.data import split_by_num, random_split, split_by_ratio, split_by_ratio_chrono

train_inter_events, eval_inter_events = split_by_ratio_chrono(inter_events, order=True, shuffle=False, test_size=0.2)
# train_data, eval_data = split_by_num(df, shuffle=True, order=False, test_size=5)
# train_inter_events, eval_inter_events = random_split(inter_events, shuffle=False, test_size=0.1)

In [258]:
inter_events.columns

Index(['item', 'category_level2_id', 'category_level1_id', 'brand_id', 'user',
       'polygon_id', 'user_item_frequency', 'user_frequency',
       'user_item_recency', 'user_recency', 'user_cat2_frequency',
       'user_cat1_frequency', 'user_cat2_recency', 'user_brand_frequency',
       'user_brand_recency', 'user_polygon_frequency', 'item_global_frequency',
       'item_polygon_frequency', 'cat2_global_frequency',
       'brand_global_frequency', 'user_distinct_items', 'user_distinct_cat2',
       'user_distinct_brands', 'time', 'day_of_week', 'week_of_year',
       'is_weekend', 'hour_bucket', 'sample_weight', 'label'],
      dtype='object')

In [259]:
exclude_cols = [
	"event_date",
	"event_hour",
	"sample_weight",
	"time",
	"label",
]

user_sparse_features = [
	# "hour_bucket",
	# 'polygon_id',
	# 'is_weekend',
]

user_dense_features = [
	# 'user_item_frequency', 'user_frequency', 'user_item_recency',
	# 'user_recency', 'user_cat2_frequency', 'user_cat1_frequency',
	# 'user_cat2_recency', 'user_brand_frequency', 'user_brand_recency',
	# 'user_polygon_frequency', 'item_global_frequency', 'item_polygon_frequency',
	# 'cat2_global_frequency', 'brand_global_frequency', 'user_distinct_items',
	# 'user_distinct_cat2', 'user_distinct_brands'
]

item_sparse_features = [
	# 'category_level2_id',
	# 'category_level1_id',
	# 'brand_id'
]

item_dense_features = [

]

# exclude_cols = [
# 	"event_date",
# 	"event_hour",
# 	"sample_weight",
# 	"time",
# 	"label",
# ]
#
# user_sparse_features = [
# ]
#
# user_dense_features = [
# ]
#
# item_sparse_features = [
# 	'category_level2_id',
# 	'category_level1_id',
# 	'brand_id'
# ]
#
# item_dense_features = [
#
# ]

In [260]:
inter_events.columns

Index(['item', 'category_level2_id', 'category_level1_id', 'brand_id', 'user',
       'polygon_id', 'user_item_frequency', 'user_frequency',
       'user_item_recency', 'user_recency', 'user_cat2_frequency',
       'user_cat1_frequency', 'user_cat2_recency', 'user_brand_frequency',
       'user_brand_recency', 'user_polygon_frequency', 'item_global_frequency',
       'item_polygon_frequency', 'cat2_global_frequency',
       'brand_global_frequency', 'user_distinct_items', 'user_distinct_cat2',
       'user_distinct_brands', 'time', 'day_of_week', 'week_of_year',
       'is_weekend', 'hour_bucket', 'sample_weight', 'label'],
      dtype='object')

In [261]:
train_inter_events = train_inter_events.groupby(['user', 'item']).agg({
	# 'sample_weight': 'sum',  # total strength
	'sample_weight': 'count',  # total strength

	'time': 'max',  # most recent interaction
	# 'order_hour': 'mean'  # most frequent hour
}).reset_index()

train_inter_events['label'] = train_inter_events['sample_weight']

In [264]:
eval_inter_events = eval_inter_events.groupby(['user', 'item']).agg({
	# 'sample_weight': 'sum',  # total strength
	'sample_weight': 'count',  # total strength

	'time': 'max',  # most recent interaction
	# 'order_hour': 'mean'  # most frequent hour        # typical hour
}).reset_index()

eval_inter_events['label'] = eval_inter_events['sample_weight']

In [265]:
train_inter_events['sample_weight'] = 1
eval_inter_events['sample_weight'] = 1

In [266]:
print(train_inter_events.head())

  user         item  sample_weight       time  label
0    1    101098711              1 2026-04-03      1
1    1     30830981              1 2025-12-27      1
2    1  30930982734              1 2026-03-23      1
3    1  31220981276              1 2026-03-23      1
4    1  31240981278              1 2026-03-23      1


In [267]:
train_data = train_inter_events[
	['user', 'item', 'time', 'label', 'sample_weight'] + user_sparse_features + user_dense_features + item_sparse_features + item_dense_features]
train_data = train_data.fillna(0)

eval_data = eval_inter_events[
	['user', 'item', 'time', 'label'] + user_sparse_features + user_dense_features + item_sparse_features + item_dense_features]
eval_data = eval_data.fillna(0)

In [268]:
print(train_data['time'].min(), train_data['time'].max())
print(eval_data['time'].min(), eval_data['time'].max())

2025-12-17 00:00:00 2026-04-15 00:00:00
2025-12-17 00:00:00 2026-04-15 00:00:00


In [269]:
# Handle single sparse columns
for c in user_sparse_features + item_sparse_features:
	for dataframe in [train_data, eval_data]:
		dataframe[c] = dataframe[c].astype(str).fillna("missing")
		dataframe.loc[dataframe[c] == "0", c] = "missing"

print("Missing value imputation done ...")

Missing value imputation done ...


In [270]:
# scaler = QuantileTransformer(output_distribution='normal', random_state=42, n_quantiles=10)
# train_data[user_dense_features + item_dense_features] = scaler.fit_transform(train_data[user_dense_features + item_dense_features]).astype(np.float32)
# eval_data[user_dense_features + item_dense_features] = scaler.transform(eval_data[user_dense_features + item_dense_features]).astype(np.float32)
#
# print("Scale done ...")

In [271]:
# train_data['sample_weight'] += 1

In [ ]:
# train_set, data_info = DatasetFeat.build_trainset(
# 	train_data=train_data,
# 	user_col=user_sparse_features + user_dense_features,
# 	item_col=item_sparse_features + item_dense_features,
# 	sparse_col=user_sparse_features + item_sparse_features,
# 	dense_col=user_dense_features + item_dense_features,
# 	# multi_sparse_col=multi_sparse_col,
# 	# pad_val=["missing", "missing"],
# 	unique_feat=False,
# 	shuffle=False
# )

# train_set, data_info = DatasetFeat.build_trainset(
# 	train_data=train_data,
# 	user_col=user_sparse_features + user_dense_features,
# 	item_col=[],
# 	sparse_col=user_sparse_features,
# 	dense_col=user_dense_features,
# 	# multi_sparse_col=multi_sparse_col,
# 	# pad_val=["missing", "missing"],
# 	unique_feat=False
# )

# eval_set = DatasetFeat.build_testset(eval_data)

from recora.data import DatasetPure

# Freeze the processed eval split so later stage-2 experiments can be compared on the same target set.
fixed_eval_data = eval_data.copy(deep=True)
fixed_eval_users = fixed_eval_data["user"].drop_duplicates().tolist()

train_set, data_info = DatasetPure.build_trainset(train_data)
eval_set = DatasetPure.build_evalset(fixed_eval_data)

print("Transformer dataset generation done ...")
print(f"Fixed eval rows: {len(fixed_eval_data)} | users: {len(fixed_eval_users)}")
# # # del train_data, eval_data

In [278]:
from recora.algorithms import (
	YouTubeRetrieval,
	YouTubeRanking,
	UserCF,
	FM,
	AutoInt,
	WideDeep,
	ItemCF,
	RNN4Rec,
	BPR,
	LightGCN,
	NGCF,
	Transformer,
	DeepFM,
	GraphSage,
	NCF,
)

# ============================================================
# Common settings
# ============================================================

METRICS = [
	"loss",
	"balanced_accuracy",
	"roc_auc",
	"pr_auc",
	"precision",
	"recall",
	"map",
	"ndcg",
]

METRICS = ["rmse", "mae", "r2"]

TF_SESS_CONFIG = {
	"gpu_options": {
		"allow_growth": True,
		"per_process_gpu_memory_fraction": 0.95,
	}
}


def reset_state(model_name: str) -> None:
	tf.compat.v1.reset_default_graph()
	print(f"\n{'=' * 30} {model_name} {'=' * 30}")


def print_hp(hp: dict) -> None:
	print("Hyperparameters:")
	print("-" * 40)
	print(hp)
	print("-" * 40)


# ============================================================
# Model hyperparameters
# ============================================================

MODEL_CONFIGS = {
	"ALS": {
		"NEG_SAMPLING": True,
		"BATCH_SIZE": 2048 * (5 + 1),

	},
	"SVD": {
		"MODEL": "SVD",
		"TASK": "rating",
		"NEG_SAMPLING": False,
		"NEG_SAMPLER": "popular+unconsumed",
		"NUM_NEG": 5,
		"BATCH_SIZE": 2048 * (5 + 1),
		"LOSS_TYPE": "focal",
		"EMBED_SIZE": 64,
		"N_EPOCHS": 10,
		"LR": 1e-3,
		"LR_DECAY": True,

	},
	"BPR": {
		"MODEL": "BPR",
		"TASK": "rating",
		"NEG_SAMPLING": False,
		"NEG_SAMPLER": "unconsumed",
		"NUM_NEG": 15,
		"BATCH_SIZE": 128 * (15 + 1),
		"LR": 1e-3,
		"LOSS_TYPE": "bpr",
		"EMBED_SIZE": 32,
		"N_EPOCHS": 3,
		"LR_DECAY": True,
	},
	"NCF": {
		"MODEL": "NCF",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "popular+unconsumed",
		"NUM_NEG": 5,
		"BATCH_SIZE": 2048 * (5 + 1),
		"LR": 4e-4,
		"LOSS_TYPE": "lambdarank",
		"TASK": "ranking",
		"EMBED_SIZE": 8,
		"HIDDEN_UNITS": (64, 32),
		"N_EPOCHS": 5,
		"LR_DECAY": True,
	},
	"DeepFM": {
		"MODEL": "DeepFM",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "popular",
		"NUM_NEG": 5,
		"BATCH_SIZE": 1024 * (5 + 1),
		"LR": 1e-4,
		"LOSS_TYPE": "bpr",
		"TASK": "ranking",
		"EMBED_SIZE": 8,
		"HIDDEN_UNITS": (256, 128, 64),
		"N_EPOCHS": 5,
		"LR_DECAY": True,
		"USE_BN": False
	},
	"TwoTower": {
		"MODEL": "TwoTower",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "popular+unconsumed",
		"NUM_NEG": 15,
		"BATCH_SIZE": 1024 * (1 + 15),
		"LR": 1e-3,
		"LOSS_TYPE": "softmax",
		"TASK": "ranking",
		"EMBED_SIZE": 8,
		"HIDDEN_UNITS": (64, 32, 16),
		"N_EPOCHS": 5,
		"LR_DECAY": True,
		"USE_BN": False
	},
	"FM": {
		"MODEL": "FM",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "popular+unconsumed",
		"NUM_NEG": 1,
		"BATCH_SIZE": 1024 * (1 + 1),
		"LR": 1e-4,
		"LOSS_TYPE": "bpr",
		"TASK": "ranking",
		"EMBED_SIZE": 8,
		"N_EPOCHS": 2,
		"LR_DECAY": True,
		"USE_BN": False
	},
	"Transformer": {
		"MODEL": "Transformer",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "unconsumed",  # random | unconsumed | popular
		"NUM_NEG": 1,
		"BATCH_SIZE": 1024 * (1 + 1),
		"LR": 2e-4,
		"LOSS_TYPE": "cross_entropy",
		"TASK": "ranking",
		"EMBED_SIZE": 16,
		"N_EPOCHS": 3,
		"LR_DECAY": True,
		"NUM_HEAD": 2,
		"NUM_TFM_LAYERS": 1,
		"HIDDEN_UNITS": (256, 128, 64),
		"RECENT_NUM": 5,
		"RANDOM_NUM": None,
	},
	"RNN4Rec": {
		"MODEL": "RNN4Rec",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "unconsumed",
		"NUM_NEG": 1,
		"BATCH_SIZE": 4096,
		"LR": 1e-3,
		"LOSS_TYPE": "bpr",
		"TASK": "ranking",
		"EMBED_SIZE": 8,
		"N_EPOCHS": 3,
		"LR_DECAY": True,
		"RECENT_NUM": 10,
		"HIDDEN_UNITS": (8, 8),
	},
	"YouTubeRetrieval": {
		"MODEL": "YouTubeRetrieval",
		"NEG_SAMPLING": True,
		"LOSS_TYPE": "sampled_softmax",
		"EMBED_SIZE": 32,
		"NORM_EMBED": False,
		"N_EPOCHS": 3,
		"LR": 1.25 * 1e-3,
		"LR_DECAY": True,
		"REG": 1e-6,
		"BATCH_SIZE": 512,
		"USE_BN": False,
		"DROPOUT_RATE": 0.2,
		"HIDDEN_UNITS": (256, 128, 64),
		"NUM_SAMPLED_PER_BATCH": 512 * (1 + 15),
		"RECENT_NUM": 31,
		"RANDOM_NUM": 31
		# multi_sparse_combiner='normal'

	},
	"YouTubeRanking": {
		"MODEL": "YouTubeRanking",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "unconsumed",
		"NUM_NEG": 3,
		"BATCH_SIZE": 2048 * (3 + 1),
		"LR": 1e-3,
		"LOSS_TYPE": "ranknet",
		"TASK": "ranking",
		"EMBED_SIZE": 32,
		"N_EPOCHS": 3,
		"LR_DECAY": True,
		"RECENT_NUM": 31,
		"RANDOM_NUM": 31,
		"HIDDEN_UNITS": (256, 128, 64),
	},
	"LightGCN": {
		"MODEL": "LightGCN",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "popular",
		"NUM_NEG": 5,
		"N_LAYERS": 3,
		"BATCH_SIZE": 1024 * (5 + 1),
		"LR": 4e-4,
		"LOSS_TYPE": "bpr",
		"TASK": "ranking",
		"EMBED_SIZE": 8,
		"N_EPOCHS": 5,
		"LR_DECAY": True,
	},
	"NGCF": {
		"MODEL": "NGCF",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "popular",
		"NUM_NEG": 1,
		"N_LAYERS": 5,
		"BATCH_SIZE": 1024 * (5 + 1),
		"LR": 8e-4,
		"LOSS_TYPE": "bpr",
		"TASK": "ranking",
		"EMBED_SIZE": 8,
		"LAYER_SIZES": (8, 8, 8),
		"NODE_DROPOUT": 0.1,
		"MESSAGE_DROPOUT": 0.1,
		"N_EPOCHS": 5,
		"LR_DECAY": True,
	},
	"GraphSage": {
		"MODEL": "GraphSage",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "unconsumed",
		"NUM_NEG": 1,
		"BATCH_SIZE": 512 * 8,
		"LR": 1e-4,
		"LOSS_TYPE": "softmax",
		"TASK": "ranking",
		"EMBED_SIZE": 16,
		"LAYER_SIZES": (16, 16),
		"HIDDEN_UNITS": (128, 64),
		"neighbor_sampling": True,
		"sample_sizes": (10, 5),
		"N_EPOCHS": 5,
		"LR_DECAY": True,
		"RECENT_NUM": 5,
		"RANDOM_NUM": 5,
	},
}

# ============================================================
# Model builder
# ============================================================
from recora.algorithms import TwoTower


def build_model(model_name: str, hp: dict, data_info):
	reset_state(model_name)

	if model_name == "BPR":
		return BPR(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			batch_size=hp["BATCH_SIZE"],
			num_neg=hp["NUM_NEG"],
		)

	if model_name == "NCF":
		return NCF(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			batch_size=hp["BATCH_SIZE"],
			use_bn=False,
			tf_sess_config=TF_SESS_CONFIG,
			num_neg=hp["NUM_NEG"],

			# reg=1e-5,
			# epsilon=1e-8,
			# dropout_rate=0.2,
			# listnet_temperature=0.1,
		)

	if model_name == 'SVD':
		from recora.algorithms import SVD
		return SVD(
            task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=True,
			reg=None,
			sampler=hp["NEG_SAMPLER"],
			num_neg=hp["NUM_NEG"],
			batch_size=hp["BATCH_SIZE"],
			lower_upper_bound=(0, 3000)
		)

	if model_name == 'TwoTower':
		return TwoTower(
			hp["TASK"],
			data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			batch_size=hp["BATCH_SIZE"],
			use_bn=hp["USE_BN"],
			hidden_units=hp["HIDDEN_UNITS"],
			tf_sess_config=TF_SESS_CONFIG,
			sampler=hp["NEG_SAMPLER"],
			num_neg=hp["NUM_NEG"],

			# epsilon=1e-8,
			# dropout_rate=0.1,
			norm_embed=False
			# reg=1e-5,
			# approx_ndcg_temperature=0.1,
			# listnet_temperature=0.1
		)

	if model_name == 'FM':
		return FM(
			hp["TASK"],
			data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			batch_size=hp["BATCH_SIZE"],
			use_bn=hp["USE_BN"],
			tf_sess_config=TF_SESS_CONFIG,
			sampler=hp["NEG_SAMPLER"],
			num_neg=hp["NUM_NEG"],

			epsilon=1e-8,
			dropout_rate=0.1,
			# reg=1e-5,
			# approx_ndcg_temperature=0.1,
			# listnet_temperature=0.1
		)

	if model_name == 'DeepFM':
		return DeepFM(
			hp["TASK"],
			data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			batch_size=hp["BATCH_SIZE"],
			use_bn=hp["USE_BN"],
			hidden_units=hp["HIDDEN_UNITS"],
			tf_sess_config=TF_SESS_CONFIG,
			sampler=hp["NEG_SAMPLER"],
			num_neg=hp["NUM_NEG"],

			epsilon=1e-8,
			dropout_rate=0.1,
			# reg=1e-5,
			# approx_ndcg_temperature=0.1,
			# listnet_temperature=0.1
		)

	if model_name == "Transformer":
		return Transformer(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			reg=None,
			batch_size=hp["BATCH_SIZE"],
			use_bn=False,
			sampler=hp["NEG_SAMPLER"],
			dropout_rate=None,
			num_heads=hp["NUM_HEAD"],
			num_tfm_layers=hp["NUM_TFM_LAYERS"],
			hidden_units=hp["HIDDEN_UNITS"],
			use_causal_mask=False,
			feat_agg_mode="concat",
			approx_ndcg_temperature=0.1,
			recent_num=hp["RECENT_NUM"],
			random_num=hp["RANDOM_NUM"],
			tf_sess_config=TF_SESS_CONFIG,
			epsilon=1e-7,
		)

	if model_name == "RNN4Rec":
		return RNN4Rec(
			task=hp["TASK"],
			data_info=data_info,
			rnn_type="gru",
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			hidden_units=hp["HIDDEN_UNITS"],
			reg=None,
			batch_size=hp["BATCH_SIZE"],
			num_neg=hp["NUM_NEG"],
			dropout_rate=None,
			recent_num=hp["RECENT_NUM"],
			tf_sess_config=TF_SESS_CONFIG,
		)

	if model_name == "YouTubeRetrieval":
		return YouTubeRetrieval(
			task="ranking",
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			norm_embed=hp["NORM_EMBED"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			reg=hp["REG"],
			batch_size=hp["BATCH_SIZE"],
			use_bn=hp["USE_BN"],
			dropout_rate=hp["DROPOUT_RATE"],
			hidden_units=hp["HIDDEN_UNITS"],
			tf_sess_config=TF_SESS_CONFIG,
			num_sampled_per_batch=hp["NUM_SAMPLED_PER_BATCH"],
			recent_num=hp["RECENT_NUM"],
			random_num=hp["RANDOM_NUM"],
			epsilon=1e-8
			# multi_sparse_combiner='normal'
		)

	if model_name == "YouTubeRanking":
		return YouTubeRanking(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			batch_size=hp["BATCH_SIZE"],
			num_neg=hp["NUM_NEG"],
			sampler=hp["NEG_SAMPLER"],
			use_bn=False,
			hidden_units=hp["HIDDEN_UNITS"],
			tf_sess_config=TF_SESS_CONFIG,
			recent_num=hp["RECENT_NUM"],
			random_num=hp["RANDOM_NUM"],
			# reg=1e-5,
			dropout_rate=0.1,
			approx_ndcg_temperature=0.1,
			listnet_temperature=0.1,
			# epsilon=1e-8
		)

	if model_name == "LightGCN":
		return LightGCN(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_layers=hp["N_LAYERS"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			batch_size=hp["BATCH_SIZE"],
			sampler=hp["NEG_SAMPLER"],
			num_neg=hp["NUM_NEG"],
			tf_sess_config=TF_SESS_CONFIG,
			# reg=1e-5,
			# epsilon=1e-8,
		)

	if model_name == "NGCF":
		return NGCF(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			layer_sizes=hp["LAYER_SIZES"],
			node_dropout_rate=hp["NODE_DROPOUT"],
			message_dropout_rate=hp["MESSAGE_DROPOUT"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			batch_size=hp["BATCH_SIZE"],
			num_neg=hp["NUM_NEG"],
			listnet_temperature=1,
			# reg=1e-5,
			# epsilon=1e-8
		)

	if model_name == "GraphSage":
		return GraphSage(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			layer_sizes=hp["LAYER_SIZES"],
			hidden_units=hp["HIDDEN_UNITS"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			batch_size=hp["BATCH_SIZE"],
			neighbor_sampling=hp["neighbor_sampling"],
			sample_sizes=hp["sample_sizes"],
			num_neg=hp["NUM_NEG"],
			sampler=hp["NEG_SAMPLER"],
			recent_num=hp["RECENT_NUM"],
			random_num=hp["RANDOM_NUM"],
			use_bn=False,
			dropout_rate=None,
			reg=None,
			listnet_temperature=0.05,
			tf_sess_config=TF_SESS_CONFIG,
			epsilon=1e-8
		)

	if model_name == "ALS":
		from recora.algorithms import ALS
		return ALS(
			task="ranking",
			data_info=data_info,
			embed_size=8,
			n_epochs=10,
			reg=1e-5,
			alpha=10
		)

	raise ValueError(f"Unsupported model: {model_name}")

In [279]:
# ============================================================
# Training entrypoint
# ============================================================

SELECTED_MODEL = "SVD"
hp = MODEL_CONFIGS[SELECTED_MODEL]

print_hp(hp)

model = build_model(SELECTED_MODEL, hp, data_info)

model.fit(
	train_data=train_set,
	neg_sampling=hp["NEG_SAMPLING"],
	verbose=2,
	shuffle=True,
	metrics=METRICS,
	eval_data=eval_set,
	eval_user_num=5000,
	k=TOP_K,
	eval_batch_size=hp["BATCH_SIZE"],
)

print_hp(hp)

Hyperparameters:
----------------------------------------
{'MODEL': 'SVD', 'TASK': 'rating', 'NEG_SAMPLING': False, 'NEG_SAMPLER': 'popular+unconsumed', 'NUM_NEG': 5, 'BATCH_SIZE': 12288, 'LOSS_TYPE': 'focal', 'EMBED_SIZE': 64, 'N_EPOCHS': 10, 'LR': 0.001, 'LR_DECAY': True}
----------------------------------------

============================== SVD ==============================
Training start time: 2026-04-16 16:24:56
With lr_decay, epoch 1 learning rate: 0.0010000000474974513


train: 100%|██████████| 1699/1699 [00:29<00:00, 56.81it/s]


Epoch 1 elapsed: 29.935s
	 train_loss: 16.8095


eval_pointwise: 100%|██████████| 75/75 [00:00<00:00, 114.71it/s]


	 eval rmse: 1.9357
	 eval mae: 0.4867
	 eval r2: 0.0496
With lr_decay, epoch 2 learning rate: 0.0009600000339560211


...

	 eval rmse: 1.6838
	 eval mae: 0.5608
	 eval r2: 0.2808
With lr_decay, epoch 10 learning rate: 0.0006925339112058282


train: 100%|██████████| 1699/1699 [00:29<00:00, 58.08it/s]


Epoch 10 elapsed: 29.256s
	 train_loss: 12.2308


eval_pointwise: 100%|██████████| 75/75 [00:00<00:00, 108.44it/s]


	 eval rmse: 1.7058
	 eval mae: 0.5702
	 eval r2: 0.2620
Hyperparameters:
----------------------------------------
{'MODEL': 'SVD', 'TASK': 'rating', 'NEG_SAMPLING': False, 'NEG_SAMPLER': 'popular+unconsumed', 'NUM_NEG': 5, 'BATCH_SIZE': 12288, 'LOSS_TYPE': 'focal', 'EMBED_SIZE': 64, 'N_EPOCHS': 10, 'LR': 0.001, 'LR_DECAY': True}
----------------------------------------


In [280]:
# model = YouTubeRetrieval(l
# 	"ranking",
# 	data_info,
# 	loss_type="sampled_softmax",
# 	embed_size=32,
# 	norm_embed=False,
# 	n_epochs=10,
# 	lr=1.25*1e-3,
# 	lr_decay=True,
# 	reg=1e-6,
# 	batch_size=512,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(256, 128, 64),
# 	tf_sess_config=config_dict,
# 	num_sampled_per_batch=512 * 15,
# 	recent_num=31,
# 	random_num=31
# 	# multi_sparse_combiner='normal'
# )

# Epoch 10 elapsed: 178.519s
# 	 train_loss: 5.8241
# eval_pointwise: 100%|██████████| 3788/3788 [00:01<00:00, 3569.97it/s]
# eval_listwise: 100%|██████████| 5000/5000 [00:05<00:00, 950.57it/s]
# 	 eval log_loss: 0.2397
# 	 eval balanced_accuracy: 0.9053
# 	 eval roc_auc: 0.9718
# 	 eval pr_auc: 0.9684
# 	 eval precision@20: 0.0295
# 	 eval recall@20: 0.2478
# 	 eval map@20: 0.1391
# 	 eval ndcg@20: 0.2134

In [281]:
# model = YouTubeRanking(
# 	"ranking",
# 	data_info,
# 	loss_type="ranknet",
# 	embed_size=32,
# 	n_epochs=5,
# 	lr=2e-3,
# 	lr_decay=False,
# 	reg=1e-6,
# 	batch_size=BATCH_SIZE,
# 	num_neg=NUM_NEG,
# 	sampler=NEG_SAMPLER,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(256, 128, 64),
# 	tf_sess_config=config_dict,
# 	recent_num=10,
# )
#
# 	 # eval log_loss: 0.3646
# 	 # eval balanced_accuracy: 0.8449
# 	 # eval roc_auc: 0.9204
# 	 # eval pr_auc: 0.7349
# 	 # eval precision@20: 0.0388
# 	 # eval recall@20: 0.1612
# 	 # eval map@20: 0.1497
# 	 # eval ndcg@20: 0.2412


In [282]:
# from libreco.data import DataInfo

# SAVE_PATH = 'youtube-retr'
# SAVE_MODEL_NAME = 'youtuberetrieval'
# # save data_info, specify model save folder
# data_info.save(
# 	path=SAVE_PATH,
# 	model_name=SAVE_MODEL_NAME
# )
# # set manual=True will use `numpy` to save model
# # set manual=False will use `tf.train.Saver` to save model
# # set inference=True will only save the necessary variables for prediction and recommendation
# model.save(
# 	path=SAVE_PATH,
# 	model_name=SAVE_MODEL_NAME,
# 	manual=False,
# 	inference_only=True
# )

In [289]:
# import numpy as np
#
# feature_names = scaler.get_feature_names_out()
# n_features = len(feature_names)
#
# # create dummy row
# x = np.zeros((1, n_features))
#
# # put your value in correct column
# idx = list(feature_names).index('order_hour')
# x[0, idx] = 8
#
# # transform
# x_scaled = scaler.transform(x)
#
# order_hour_scaled = x_scaled[0, idx]
# print(order_hour_scaled)

result = model.recommend_user(
	"30953879",
	n_rec=630,
	filter_consumed=False,
	# user_feats={
	# 	"order_hour": 0,
	# }
)

result = pd.DataFrame(result)
result[['category_id', 'brand_id']] = (
	result["30953879"]
	.str.split("098", expand=True, n=1)  # n=1 ensures we only split once
)

brand_category_titles = pd.read_csv('./sample_data/brand_category_titles.csv')
brand_category_titles['category_id'] = brand_category_titles['category_id'].astype(str)
brand_category_titles['brand_id'] = brand_category_titles['brand_id'].astype(str)
result = result.merge(brand_category_titles, on=['category_id', 'brand_id'], how='left')
result[result['category_id'] == "3085"]
# result

,30953879,category_id,brand_id,category_title,brand_title
29,3085098250,3085,250,چیپس,مزمز
78,3085098253,3085,253,چیپس,چی‌توز
134,30850982269,3085,2269,چیپس,باتو
214,3085098286,3085,286,چیپس,پرینگلز
227,3085098256,3085,256,چیپس,ترددیلا
318,30850989065,3085,9065,چیپس,یامی
553,3085098196,3085,196,چیپس,لینا
564,3085098207,3085,207,چیپس,متفرقه


In [96]:
result = train_data[train_data['user'] == "30953879"].groupby(['item'])['sample_weight'].count().reset_index()
brand_category_titles['category_id'] = brand_category_titles['category_id'].astype(str)
brand_category_titles = brand_category_titles.drop_duplicates(subset=['category_id', 'category_title'])
result.merge(brand_category_titles, left_on=['item'], right_on=['category_id'], how='left').sort_values(by='sample_weight', ascending=False)

,item,sample_weight,category_id,brand_id,category_title,brand_title
16,651,25,651,11053,سیگار,اونیکس
0,101,17,101,604,نوشیدنی انرژی‌زا,هوفنبرگ
5,3085,4,3085,16895,چیپس,کرانچیپس
12,3133,4,3133,6681,کنسرو سبزیجات و حبوبات,شایان
13,33,4,33,10168,ویفر و بیسکویت,لالائی
19,69,3,69,16276,شیر,سوت چلیر
25,98,2,98,9844,قهوه و هات‌چاکلت,مولتی کوکو
24,96,2,96,13013,دوغ,توریس
8,3100,2,3100,805,ماکارونی، رشته و لازانیا,ترخینه
18,68,2,68,14373,خامه,آتلیش


In [89]:
result = df.loc[df['user'] == "30953879"]
result['category_id'] = pd.to_numeric(result['item'], errors='coerce')
brand_category_titles = pd.read_csv('./sample_data/brand_category_titles.csv').drop_duplicates(subset=['category_id'])
result = result.merge(brand_category_titles, on=['category_id'], how='left')
result[['sample_weight', 'category_id', 'category_title']].sort_values(by='sample_weight', ascending=False)

,sample_weight,category_id,category_title
111,0.820335,101,نوشیدنی انرژی‌زا
110,0.820335,62,غذای نیمه آماده منجمد
109,0.742997,101,نوشیدنی انرژی‌زا
108,0.742997,101,نوشیدنی انرژی‌زا
107,0.742997,3082,آدامس و خوشبو کننده دهان
...,...,...,...
4,0.021030,3082,آدامس و خوشبو کننده دهان
3,0.021030,651,سیگار
2,0.017251,3133,کنسرو سبزیجات و حبوبات
1,0.017251,651,سیگار


In [ ]:
import tensorflow as tf
from libreco.data import DataInfo
from libreco.algorithms import TwoTower, YouTubeRetrieval

# important to reset graph if model is loaded in the same shell.
tf.compat.v1.reset_default_graph()
# load data_info
data_info = DataInfo.load(
	SAVE_PATH, model_name=SAVE_MODEL_NAME
)
print(data_info)
# load model, should specify the model name, e.g., DeepFM
model: YouTubeRetrieval = YouTubeRetrieval.load(
	path=SAVE_PATH, model_name=SAVE_MODEL_NAME, data_info=data_info, manual=True
)

In [ ]:
# model.user_embeds_np[data_info.user2id[30953879]].shape
model.user_embeds_np.shape

In [ ]:
import faiss
import numpy as np
import pandas as pd

faiss_index: faiss.swigfaiss.IndexIVFFlat = faiss.read_index('./embed_model/faiss_index.bin')
user_embedding = model.get_user_embedding(30953879, include_bias=True).astype(np.float32).reshape(1, -1)
distances, ids = faiss_index.search(user_embedding, 100)

In [ ]:
distances

In [ ]:
result = pd.DataFrame(list(map(lambda item: list(map(lambda x: int(x), item.split("012"))), [data_info.id2item[x] for x in ids.ravel().tolist()])),
                      columns=['brand_id', 'category_id'])
result['distances'] = distances.ravel()

brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')

In [ ]:
# result[result['category_id'] == 69]
result

# Init `embed_deploy`

In [ ]:
from libserving.serialization import embed2redis, save_embed, save_faiss_index

path = "embed_model"  # specify model saving directory
save_embed(path, model)  # save model in json format

In [ ]:
from libserving.serialization.redis import redis_connection, model_name2redis, id_mapping2redis, user_consumed2redis, user_embed2redis
import ujson
import os

host = "localhost"
port = 6379
db = 0
chunk_size = 10000
with redis_connection(host, port, db) as r:
	model_name2redis(path, r)
	print("\nmodel_name2redis ...")

with redis_connection(host, port, db) as r:
	id_mapping2redis(path, r)
	print("\nid_mapping2redis ...")

with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_consumed.json")) as f:
		data = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, items) in enumerate(data.items()):
		pipe.hset("user_consumed", u, ujson.dumps(items))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_consumed2redis ...")

with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_embed.json")) as f:
		user_embeds = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, embed) in enumerate(user_embeds.items()):
		pipe.hset("user_embed", u, ujson.dumps(embed))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_embed2redis ...")

In [ ]:
from libserving.serialization import save_faiss_index

print(path, model)
save_faiss_index(path, model)

In [ ]:
import requests
import json

payload = {
	"user": "30953879",
	"n_rec": 100
}

result = requests.post(
	"http://127.0.0.1:8000/embed/recommend",
	headers={"Content-Type": "application/json"},
	data=json.dumps(payload)
).json()

In [ ]:
result[['brand_id', 'category_id']] = (
	result['item']
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)

# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')

# Init `online_deploy`

In [ ]:
from libserving.serialization import save_online, online2redis

path = "online_model"  # specify model saving directory
save_online(path, model, version=1)  # save model in json format

In [ ]:
from libserving.serialization.redis import redis_connection, model_name2redis, id_mapping2redis
import ujson
import os

host = "localhost"
port = 6379
db = 0
chunk_size = 10000  # ← same as you used before

# ------------------------------------------------------------------
# 1. model_name + id_mapping (small, unchanged)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	model_name2redis(path, r)
	print("\nmodel_name2redis ...")

with redis_connection(host, port, db) as r:
	id_mapping2redis(path, r)
	print("\nid_mapping2redis ...")

# ------------------------------------------------------------------
# 2. user_consumed (your existing chunked version)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_consumed.json")) as f:
		data = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, items) in enumerate(data.items()):
		pipe.hset("user_consumed", u, ujson.dumps(items))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_consumed2redis ...")

# ------------------------------------------------------------------
# 3. features2redis → FULLY CHUNKED (the heavy part for online models)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	feature_path = os.path.join(path, "features.json")
	with open(feature_path) as f:
		feats = ujson.load(f)

	r.set("n_users", feats["n_users"])
	r.set("n_items", feats["n_items"])
	if "max_seq_len" in feats:
		r.set("max_seq_len", feats["max_seq_len"])

	# user_sparse_values (per-user)
	if "user_sparse_col_index" in feats:
		r.hset("feature", "user_sparse", 1)
		r.set("user_sparse_col_index", ujson.dumps(feats["user_sparse_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["user_sparse_values"]):
			pipe.hset("user_sparse_values", str(i), ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# item_sparse_values (list push)
	if "item_sparse_col_index" in feats:
		r.hset("feature", "item_sparse", 1)
		r.set("item_sparse_col_index", ujson.dumps(feats["item_sparse_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["item_sparse_values"]):
			pipe.rpush("item_sparse_values", ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# user_dense_values (per-user)
	if "user_dense_col_index" in feats:
		r.hset("feature", "user_dense", 1)
		r.set("user_dense_col_index", ujson.dumps(feats["user_dense_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["user_dense_values"]):
			pipe.hset("user_dense_values", str(i), ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# item_dense_values (list push)
	if "item_dense_col_index" in feats:
		r.hset("feature", "item_dense", 1)
		r.set("item_dense_col_index", ujson.dumps(feats["item_dense_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["item_dense_values"]):
			pipe.rpush("item_dense_values", ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	print("\nfeatures2redis ...")

# ------------------------------------------------------------------
# 4. user_sparse2redis (small, no chunk needed)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	user_sparse_fields_path = os.path.join(path, "user_sparse_fields.json")
	if os.path.exists(user_sparse_fields_path):
		with open(user_sparse_fields_path) as f:
			user_sparse_fields = ujson.load(f)
		r.hset("user_sparse_fields", mapping=user_sparse_fields)

	user_sparse_idx_mapping_path = os.path.join(path, "user_sparse_idx_mapping.json")
	if os.path.exists(user_sparse_idx_mapping_path):
		with open(user_sparse_idx_mapping_path) as f:
			user_sparse_idx_mapping = ujson.load(f)
		for col, idx_mapping in user_sparse_idx_mapping.items():
			col_name = f"user_sparse_idx_mapping__{col}"
			r.hset(col_name, mapping=idx_mapping)

	print("\nuser_sparse2redis ...")

# ------------------------------------------------------------------
# 5. user_dense2redis (small, no chunk needed)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	user_dense_fields_path = os.path.join(path, "user_dense_fields.json")
	if os.path.exists(user_dense_fields_path):
		with open(user_dense_fields_path) as f:
			user_dense_fields = ujson.load(f)
		r.hset("user_dense_fields", mapping=user_dense_fields)

	print("\nuser_dense2redis ...")

print("\n✅ All online2redis data loaded into Redis with chunking!")

In [ ]:
from libserving.serialization import save_faiss_index

save_faiss_index(path, model)

In [ ]:
result = model.recommend_user(
	30953879,
	n_rec=10,
	filter_consumed=False,
	# user_feats={
	# 	"order_hour": 20
	# }
)

result = pd.DataFrame(result)
result[['brand_id', 'category_id']] = (
	result[30953879]
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)
# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result

In [ ]:
json.dumps(payload)

In [ ]:
import requests
import json

payload = {
	"user": 30953879,
	"n_rec": 10,
	"filter_consumed": False,
	"return_scores": False,
	"user_feats": {
		"order_hour": 12
	}
}

result = requests.post(
	"http://127.0.0.1:8000/online/recommend",
	headers={"Content-Type": "application/json"},
	data=json.dumps(payload)
).json()

result = pd.DataFrame(result)
result[['brand_id', 'category_id']] = (
	result['recommendations']
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)

# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result

### Read `brand_category_titles` data

In [ ]:
BRAND_CATEGORY_SQL = """
SELECT category_level2_id                                        AS category_id,
       brand_id                                                  AS brand_id,
       dictGet('KF.dim_categories', 'title', category_level2_id) AS category_title,
       dictGet('KF.dim_brands', 'title', brand_id)               AS brand_title
FROM KF.plane_products
GROUP BY category_level2_id, brand_id
"""

brand_category_titles = client.query_df(BRAND_CATEGORY_SQL)

In [ ]:
brand_category_titles.to_csv('brand_category_titles.csv', index=False)

In [ ]:
brand_category_titles = pd.read_csv('brand_category_titles.csv')